In [0]:
df = spark.sql(f"SELECT * FROM ecommerce.gold.feature_engineered_data").toPandas()

In [0]:
df.head()
df.shape

In [0]:
X = df.drop([
    "order_id",
    "customer_id",
    "order_status",
    "purchase_date",
    "estimated_delivery_date",
    "delivered_date",
    "delivery_days",
    "is_late_delivery"
], axis=1)

y = df["is_late_delivery"]

In [0]:
X.dtypes

In [0]:
df["customer_state"].nunique()

In [0]:
city_freq = X["customer_city"].value_counts(normalize=True)

X["customer_city_freq"] = X["customer_city"].map(city_freq)

seller_city_freq = df["seller_city"].value_counts(normalize=True)

X["seller_city_freq"] = df["seller_city"].map(seller_city_freq)

X.drop(["customer_city","seller_city"], axis=1, inplace=True)

In [0]:
import pandas as pd

X = pd.get_dummies(
    X,
    columns=["customer_state", "seller_state"],
    drop_first=True
)

In [0]:
X.dtypes

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [0]:
import mlflow
import mlflow.sklearn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedShuffleSplit, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve
)


In [0]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=0.3,   # creates 30% minority class
    random_state=42
)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
y_train.value_counts(normalize=True)

In [0]:


# 1. Stratified Subsample for Tuning


sss = StratifiedShuffleSplit(n_splits=1, test_size=0.8, random_state=42)

for train_index, _ in sss.split(X_train, y_train):
    X_sub = X_train.iloc[train_index]
    y_sub = y_train.iloc[train_index]

# 2. Hyperparameter Search
param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}

base_model =XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.8,
    gamma=0.2,
    min_child_weight=5
)

search = RandomizedSearchCV(
    base_model,
    param_grid,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    n_jobs=-1
)

search.fit(X_sub, y_sub)

best_params = search.best_params_

In [0]:
# 3. Train Final Model on Full Data
with mlflow.start_run():

    model = XGBClassifier(
        **best_params,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_train, y_train)
    # 4. Predict Probabilities
   
    y_prob = model.predict_proba(X_test)[:,1]

In [0]:
 # 5. Threshold Optimization

precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_prob)

f1_scores = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]

y_pred = (y_prob >0.4).astype(int)

In [0]:
 # 6. Metrics

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

# 7. Log to MLflow

mlflow.log_params(best_params)
mlflow.log_param("best_threshold", best_threshold)

mlflow.log_metric("accuracy", acc)
mlflow.log_metric("precision", precision)
mlflow.log_metric("recall", recall)
mlflow.log_metric("f1_score", f1)
mlflow.log_metric("roc_auc", auc)

cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

mlflow.log_figure(plt.gcf(), "confusion_matrix.png")
plt.close()


mlflow.sklearn.log_model(model, "delivery_delay_model")